# Module 3.2 — Turbulence Models

**The central engineering problem:**
At high Reynolds number, directly simulating every eddy costs $Re^3$ — impossible for real flows. Instead we **model** the effect of turbulence on the mean flow. This module explains how, physically and mathematically — and tells you which model to choose for which problem.

**Roadmap:**
1. Eddy viscosity — physics first (why turbulence acts like viscosity)
2. The $k$-$\varepsilon$ model — the factory analogy, every term explained
3. The $k$-$\omega$ SST model — why a new model was needed, and the blending idea
4. Wall functions — the engineering shortcut and its limits
5. Spalart-Allmaras — one equation is enough for aerodynamics
6. Implementation — turbulent channel flow with mixing-length model
7. Model selection guide

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

## 1. Eddy Viscosity — Physics First

### Why turbulence acts like viscosity

In laminar flow, molecular collisions transfer momentum between fluid layers. The flux is proportional to the velocity gradient:
$$\tau_{\text{lam}} = \rho\nu\frac{d\bar{u}}{dy}$$

In turbulent flow, fluid parcels from fast-moving regions are flung into slow regions and vice versa. This turbulent momentum exchange is **much more efficient** than molecular collisions — same mathematical form, much larger effective viscosity:
$$\tau_{\text{turb}} = \rho\nu_t\frac{d\bar{u}}{dy}$$

| Property | Molecular $\nu$ | Eddy $\nu_t$ |
|----------|-----------------|---------------|
| Property of | The **fluid** | The **flow** |
| Typical value | $10^{-6}$ m²/s (water) | $10^{-2}$ m²/s (turbulent pipe) |
| Ratio $\nu_t/\nu$ | 1 (by definition) | $10^3$–$10^4$ near pipe centre |
| Varies in space? | No | Yes — strongly |
| Known before solving? | ✅ Yes | ❌ No — must be modelled |

### The eddy viscosity formula for $k$-$\varepsilon$

$$\boxed{\nu_t = C_\mu \frac{k^2}{\varepsilon}}$$

**Why $k^2/\varepsilon$?** From dimensional analysis:
- $k$ [m²/s²]: turbulent kinetic energy — a velocity scale squared
- $\varepsilon$ [m²/s³]: dissipation rate — energy per unit time
- $k/\varepsilon$ [s]: turbulent **time scale** — how long an eddy lives
- $\sqrt{k}$ [m/s]: turbulent **velocity scale** — how fast eddies move
- $\nu_t \sim $ velocity × length = $\sqrt{k} \cdot k^{3/2}/\varepsilon = k^2/\varepsilon$ ✓

$C_\mu = 0.09$ is calibrated from homogeneous isotropic turbulence experiments.

## 2. The $k$-$\varepsilon$ Model — The Factory Analogy

### Physical picture: $k$ as inventory

Think of turbulent kinetic energy $k$ as **inventory in a factory**:

```
Mean flow shear → PRODUCTION Pk → eddies cascade → DISSIPATION ε → heat
  (raw material)     (inventory)    (assembly line)   (waste product)

Factory balance: k increases when Pk > ε, decreases when Pk < ε
Steady state (equilibrium turbulence): Pk = ε exactly
```

### Transport equation for $k$ — every term explained

$$\underbrace{\frac{Dk}{Dt}}_{\text{following parcel}} = \underbrace{P_k}_{\text{production}} - \underbrace{\varepsilon}_{\text{destruction}} + \underbrace{\frac{\partial}{\partial x_j}\left[(\nu + \nu_t/\sigma_k)\frac{\partial k}{\partial x_j}\right]}_{\text{diffusion}}$$

- **$P_k = \nu_t|\nabla\bar{u}|^2$**: turbulence is produced where there is mean shear. Near a wall or in a jet shear layer, shear is large → $P_k$ is large.
- **$\varepsilon$**: at the Kolmogorov scale, viscosity converts eddy energy to heat. Note: this is NOT a loss from the flow as a whole — it heats the fluid slightly.
- **Diffusion**: turbulence spreads spatially. $\sigma_k = 1.0$ is the turbulent Prandtl number for $k$.

### Transport equation for $\varepsilon$ — every term explained

$$\frac{D\varepsilon}{Dt} = \underbrace{C_{\varepsilon 1}\frac{\varepsilon}{k}P_k}_{\text{production of dissipation}} - \underbrace{C_{\varepsilon 2}\frac{\varepsilon^2}{k}}_{\text{destruction of dissipation}} + \frac{\partial}{\partial x_j}\left[(\nu + \nu_t/\sigma_\varepsilon)\frac{\partial\varepsilon}{\partial x_j}\right]$$

- **Why $\varepsilon/k$ multiplies $P_k$?** The turbulent time scale is $k/\varepsilon$. $\varepsilon$ is produced at a rate proportional to $P_k$ and inversely proportional to eddy lifetime.
- **Why $\varepsilon^2/k$?** Self-destruction — dissipation destroys itself. The $k$ is the timescale normalisation.

### Standard constants

| $C_\mu$ | $C_{\varepsilon 1}$ | $C_{\varepsilon 2}$ | $\sigma_k$ | $\sigma_\varepsilon$ |
|---------|---------------------|---------------------|------------|---------------------|
| 0.09    | 1.44                | 1.92                | 1.0        | 1.3                 |

All calibrated from simple test flows (channel, grid turbulence). Surprisingly robust.

### Known failure: near-wall behaviour

As $y \to 0$, $\varepsilon$ must satisfy $\varepsilon = 2\nu(\partial\sqrt{k}/\partial y)^2$ (exact from N-S). This requires $y^+ < 1$ meshing with special low-Re damping. **Standard $k$-$\varepsilon$ without damping breaks near walls** — do not use it for wall-bounded flows without wall functions.

## 3. $k$-$\omega$ SST — Why a New Model Was Needed

### The problem with $k$-$\varepsilon$ near walls

Replace $\varepsilon$ with $\omega = \varepsilon/(C_\mu k)$ — the **specific dissipation rate** [1/s]: how many times per second does an eddy lose its energy.

Near the wall: $\omega$ has a known analytical boundary condition. This means $k$-$\omega$ does not need special damping near walls — a huge practical advantage.

**But** $k$-$\omega$ is sensitive to the free-stream value of $\omega_\infty$ — small changes in what you specify far from walls give very different results.

### The SST blending idea (Menter 1994)

**Key insight:** $k$-$\omega$ is better **near walls**; $k$-$\varepsilon$ is better **away from walls**. Why not use both — blend them based on distance from the wall?

$$\phi_{\text{SST}} = F_1 \cdot \phi_{k\text{-}\omega} + (1-F_1) \cdot \phi_{k\text{-}\varepsilon}$$

where $F_1 = 1$ (use $k$-$\omega$) near the wall and $F_1 = 0$ (use $k$-$\varepsilon$) in the free stream.

### SST also adds a stress limiter

Standard $k$-$\varepsilon$ over-predicts $\nu_t$ in **stagnation regions** (point in front of a cylinder). The shear $|\nabla\bar{u}|$ is large there → $P_k$ is large → too much $k$.

SST adds a limiter using the rotation rate $\Omega$:
$$\nu_t = \frac{a_1 k}{\max(a_1\omega,\, \Omega F_2)}$$

When rotation dominates (stagnation), $\nu_t$ is reduced. This is why SST predicts drag and lift on airfoils much better than standard $k$-$\varepsilon$.

| Behaviour | $k$-$\varepsilon$ | $k$-$\omega$ SST |
|-----------|------------------|------------------|
| Near-wall accuracy | ❌ Needs damping | ✅ Well-posed BC |
| Free-stream sensitivity | ✅ Insensitive | ⚠️ Fixed by blending |
| Stagnation regions | ❌ Over-predicts $\nu_t$ | ✅ Stress limiter corrects |
| Separated flows | ❌ Poor | ✅ Better (not perfect) |
| OpenFOAM/Fluent default | No | ✅ Yes |

## 4. Wall Functions — The Engineering Shortcut

### Why the viscous sublayer is expensive

The viscous sublayer extends to $y^+ < 5$. In physical units:

$$y_1 = \frac{5\nu}{u_\tau}$$

For a pipe at Re = $10^6$: $u_\tau \approx 0.05$ m/s, $\nu = 10^{-6}$ m²/s → $y_1 = 0.1$ mm.

If the pipe is 1 m in diameter, you need cells 0.1 mm thick in a 1000 mm domain — a ratio of 1:10,000. Industrial meshes cannot afford this everywhere.

### The wall function idea

Place the first cell in the **log-law region** ($y^+ \approx 30$–100). Assume the flow in that cell follows the log-law and use it analytically to compute the wall shear stress:

$$u^+ = \frac{1}{\kappa}\ln(y^+) + B \implies \text{solve for } u_\tau \implies \tau_w = \rho u_\tau^2$$

### BCs for $k$ and $\varepsilon$ in the wall-adjacent cell

Under equilibrium assumption ($P_k = \varepsilon$):

$$k_w = \frac{u_\tau^2}{\sqrt{C_\mu}}, \qquad \varepsilon_w = \frac{u_\tau^3}{\kappa y_1}$$

### The critical y+ rule

| First cell location | Behaviour | Recommendation |
|---------------------|-----------|----------------|
| $y^+ < 5$ (sublayer) | Resolves physics accurately | Use low-Re SST |
| $5 < y^+ < 30$ (buffer) | **Worst region** — neither law applies | ❌ **Avoid** — results unreliable |
| $y^+ = 30$–100 (log-law) | Wall function valid | Use standard wall functions |
| $y^+ > 200$ | Too coarse — misses near-wall physics | Refine the mesh |

## 5. Spalart-Allmaras — One Equation for Aerodynamics

SA solves **one transport equation** for a modified eddy viscosity $\tilde{\nu}$:

$$\frac{D\tilde{\nu}}{Dt} = \underbrace{c_{b1}\tilde{S}\tilde{\nu}}_{\text{production}} - \underbrace{c_{w1}f_w\left(\frac{\tilde{\nu}}{d}\right)^2}_{\text{near-wall destruction}} + \underbrace{\frac{1}{\sigma}\nabla\cdot[(\nu+\tilde{\nu})\nabla\tilde{\nu}]}_{\text{diffusion}}$$

where $d$ is the **distance to the nearest wall** — built explicitly into the model.

The actual eddy viscosity has a near-wall damping:
$$\nu_t = \tilde{\nu}f_{v1}, \qquad f_{v1} = \frac{\chi^3}{\chi^3 + c_{v1}^3}, \quad \chi = \tilde{\nu}/\nu$$

- $\chi \ll 1$ (near wall): $f_{v1} \approx 0$ → $\nu_t \approx 0$ — viscous sublayer is naturally captured
- $\chi \gg 1$ (away from wall): $f_{v1} \approx 1$ → $\nu_t \approx \tilde{\nu}$

**When to use SA:** attached boundary layers on aerodynamic surfaces. Fast, robust, well-validated for attached BL flows. The basis for DES (Detached Eddy Simulation).

In [ ]:
# ── Turbulent channel flow with mixing-length model ──────────────────────────
# Setup:
#   Fully-developed channel between y=0 (wall) and y=H (symmetry)
#   Pressure gradient -dP/dx drives the flow
#   RANS momentum: d/dy[(ν+νt)*du/dy] = (1/ρ)*dP/dx
#   Mixing-length with Van Driest damping: ℓm = κy*(1-exp(-y+/26))
#   νt = ℓm² |du/dy|
#   Iteration: guess u → compute νt → solve linear system → update u_τ

# ── Model constants ──────────────────────────────────────────────────────────
C_mu  = 0.09
kappa = 0.41

# ── Flow parameters (targeting Re_τ ≈ 395, classic DNS benchmark) ────────────
nu   = 1e-5    # kinematic viscosity [m²/s]
dPdx = -1.248e-4  # pressure gradient [Pa/m]
rho  = 1.0     # density [kg/m³]
H    = 1.0     # half-channel height [m]

# ── Grid: fine enough to resolve viscous sublayer (y+ ≈ 0.8 for first cell) ──
N  = 500
y  = np.linspace(0, H, N)
dy = y[1] - y[0]

def mixing_length(y, u_tau, nu):
    """ℓm = κy·(1 - exp(-y+/26)) — Van Driest damped mixing length"""
    y_plus = y * u_tau / nu
    return kappa * y * (1 - np.exp(-y_plus / 26.0))

def solve_channel(nu_eff, dPdx, rho, dy, N):
    """
    Solve: d/dy[(nu_eff)*du/dy] = dPdx/rho
    BCs: u[0]=0 (wall no-slip), du/dy=0 at y=H (symmetry)
    Banded tridiagonal system — same structure as BTCS diffusion.
    """
    nu_face = 0.5*(nu_eff[:-1] + nu_eff[1:])   # N-1 face values

    diag = np.zeros(N)
    rhs  = np.full(N, dPdx/rho)

    diag[1:-1] = -(nu_face[:-1] + nu_face[1:]) / dy**2

    ab      = np.zeros((3, N))
    ab[0,1:]  = nu_face / dy**2       # super-diagonal
    ab[1,:]   = diag
    ab[2,:-1] = nu_face / dy**2       # sub-diagonal

    # Wall BC: u[0]=0
    ab[1, 0]  = 1.0;  ab[0, 1]  = 0.0;  rhs[0] = 0.0
    # Symmetry BC: du/dy=0 at y=H → one-sided difference
    ab[1, -1] = -nu_face[-1] / dy**2;  ab[2, -2] = nu_face[-1] / dy**2
    rhs[-1]   = dPdx / rho

    return solve_banded((1, 1), ab, rhs)

# ── Picard iteration ──────────────────────────────────────────────────────────
u_bar = np.zeros(N)
u_tau = 0.004   # initial guess

for iteration in range(500):
    lm      = mixing_length(y, u_tau, nu)
    S       = np.gradient(u_bar, y)          # du/dy
    nu_t    = lm**2 * np.abs(S)              # eddy viscosity
    nu_eff  = nu + nu_t                       # total viscosity

    u_bar   = solve_channel(nu_eff, dPdx, rho, dy, N)

    tau_w   = rho * nu_eff[1] * (u_bar[1] - u_bar[0]) / dy
    u_tau   = np.sqrt(abs(tau_w) / rho)

y_plus  = y * u_tau / nu
Re_tau  = u_tau * H / nu
u_plus  = u_bar / u_tau

print(f'Converged: u_τ = {u_tau:.5f} m/s   Re_τ = {Re_tau:.1f}   (target: 395)')
print(f'First cell: y+ = {y_plus[1]:.2f}  (should be < 1 for sublayer resolution)')
print(f'Max u+ = {u_plus.max():.2f}  (centreline velocity in wall units)')

In [ ]:
# ── Plots: law of the wall + profile comparison ───────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: law of the wall comparison ─────────────────────────────────────────
yp_theory = np.logspace(-0.3, 3.0, 500)

ax1.semilogx(y_plus[1:], u_plus[1:], 'b-', lw=2.5, label='Simulation (mixing-length)')
ax1.semilogx(yp_theory[yp_theory <= 11.6], yp_theory[yp_theory <= 11.6],
             'k--', lw=1.5, label='$u^+ = y^+$ (viscous sublayer)')
ax1.semilogx(yp_theory[yp_theory >= 30],
             (1/kappa)*np.log(yp_theory[yp_theory >= 30]) + 5.2,
             'r--', lw=1.5, label='Log-law: $u^+ = (1/\\kappa)\\ln(y^+)+5.2$')
ax1.axvspan(0.3,   5,  alpha=0.10, color='blue',  label='Viscous sublayer')
ax1.axvspan(30,  200,  alpha=0.10, color='red',   label='Log-law region')
ax1.set_xlim([0.3, min(800, y_plus[-1])])
ax1.set_ylim([0, 30])
ax1.set_xlabel('$y^+$', fontsize=12); ax1.set_ylabel('$u^+$', fontsize=12)
ax1.set_title('Law of the Wall\nBlue line should follow u+=y+ then log-law')
ax1.legend(fontsize=8); ax1.grid(True, which='both', alpha=0.3)

# ── Right: compare turbulent vs laminar profile shape ────────────────────────
u_lam     = -dPdx / (2*rho*nu) * y * (2*H - y)   # laminar Poiseuille
u_bar_max = u_bar[-1];  u_lam_max = u_lam[-1]

ax2.plot(u_bar / u_bar_max, y, 'b-', lw=2.5,
         label=f'Turbulent (u_max = {u_bar_max:.2f} m/s)')
ax2.plot(u_lam / u_lam_max, y, 'g--', lw=2,
         label=f'Laminar Poiseuille (u_max = {u_lam_max:.1f} m/s)')
ax2.set_xlabel('$u/u_{max}$ (normalised)', fontsize=11)
ax2.set_ylabel('$y$ [m]', fontsize=11)
ax2.set_title('Profile Shape\nTurbulent profile is flatter — strong near-wall mixing')
ax2.legend(fontsize=9); ax2.grid(True)
ax2.set_xlim([0, 1.05])

plt.suptitle(f'Turbulent Channel: $u_\\tau = {u_tau:.4f}$ m/s, $Re_\\tau = {Re_tau:.0f}$',
             fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nHow to read the left plot:')
print(f'  For y+ < 5:  simulation should follow u+ = y+ (viscous sublayer)')
print(f'  For y+ > 30: simulation should follow log-law u+ = (1/κ)ln(y+) + 5.2')
print(f'  Deviation near centreline (y+ = {y_plus[-1]:.0f}) is expected')
print(f'  — log-law assumes infinite wall, not valid near the centreline')
print(f'\nRight plot: turbulent profile is much flatter than laminar (parabolic).')
print(f'  Strong near-wall mixing homogenises the core velocity.')
print(f'  For same pressure gradient: turbulent u_max = {u_bar_max:.2f}, laminar = {u_lam_max:.1f}')
print(f'  → Turbulence dramatically reduces the centreline velocity (more resistance).')

## 6. Model Selection Guide

| Flow type | Recommended model | Reason |
|-----------|------------------|---------|
| Attached BL, aerospace | Spalart-Allmaras | Fast, well-validated for attached flows |
| General engineering, adverse pressure gradient | $k$-$\omega$ SST | Best near-wall + free-stream blend |
| Free shear flows (jets, wakes) | Realizable $k$-$\varepsilon$ | Better for free-stream without walls |
| Buoyancy-driven flows | $k$-$\varepsilon$ RNG | Better for weak shear, buoyancy |
| Strong swirl, rotation | Reynolds Stress Model (RSM) | Boussinesq fails for anisotropic cases |
| Unsteady separated flows | LES or DES | RANS smears unsteady structures |

### The Boussinesq assumption — when it fails

All eddy-viscosity models assume turbulence is **isotropic** and **aligned with mean strain**. This fails for:
- **Strong curvature** (bent pipes): normal stresses become unequal
- **Rapid rotation** (turbomachinery): Coriolis makes turbulence anisotropic
- **Impingement** (jet hitting a wall): stagnation region behaviour is wrong

For these cases, use the Reynolds Stress Model (RSM) which solves transport equations for all 6 stress components directly.

## Summary

| Model | Equations | $\nu_t$ | Strength | Weakness |
|-------|-----------|---------|---------|----------|
| Mixing-length | 0 | $\ell_m^2|d\bar{u}/dy|$ | Simple, pedagogical | 1D shear only |
| SA | 1 ($\tilde{\nu}$) | $\tilde{\nu}f_{v1}$ | Aerospace BL, fast | Free shear poor |
| $k$-$\varepsilon$ | 2 ($k,\varepsilon$) | $C_\mu k^2/\varepsilon$ | Free jets, wakes | Near-wall singular |
| $k$-$\omega$ SST | 2 ($k,\omega$) | Blended | **Industry default** | Complex geometry |
| RSM | 7 stress eqs | Direct | Anisotropic, swirl | Expensive, fragile |

---
**Next:** Module 3.3 — Mesh Generation: structured vs unstructured, quality metrics, near-wall meshing for turbulent flows.

## Exercise

**Predict first, then verify.**

1. Change `dPdx = -5e-4` in the channel solver (stronger pressure gradient). **Predict first:** does $Re_\tau$ increase or decrease? Does the log-law region shift to higher $y^+$ values? Run and compare.

2. Change `nu = 1e-4` (10× more viscous fluid — like heavy oil instead of air). **Predict:** does the turbulent profile become more or less parabolic? Explain in terms of the $Re_\tau$ and the relative thickness of the viscous sublayer.

3. Replace the Van Driest damping with simple mixing length $\ell_m = \kappa y$ (no damping). What happens to the velocity profile near the wall ($y^+ < 10$)? Plot both profiles on the same axis and identify where they diverge.